# Baseline: astronaut health anomaly detection (practice run)

NASA Space Apps 2026 · *Create Health Monitoring Software for Astronauts on Space Missions*

This notebook:

1. generates synthetic daily health data for a fictional 4-person crew,
2. runs our personal-baseline engine (a Python copy of the app's `baseline.js`),
3. trains an Isolation Forest model,
4. compares both with plain Earth range rules,
5. exports `crew.json` and the trained model for the app.

**About the data:** it is synthetic, not real NASA records. No public dataset has daily vitals for astronauts. The numbers are built from published figures, such as the ~6 h average sleep on the ISS and the 3-minute reaction test used there, and known events are planted so we can measure what each method catches.

Run everything with **Runtime → Run all**. It takes under a minute.

In [ ]:
# Setup. Colab already has all of these installed.
import json
from datetime import date, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest

try:
    from IPython.display import display, Markdown
except ImportError:  # running outside a notebook
    display, Markdown = print, str

pd.set_option("display.width", 160)
print("Ready")

## 1. Generate synthetic crew data

Each crew member gets their own resting levels, plus daily noise. Everyone loses about 0.6 h of sleep over the 60 days, a slow drift toward the ISS average. Three events are planted, and the fourth crew member is a control with no events, for counting false alarms.

| Crew | Days | Event |
| --- | --- | --- |
| crew-01 | 22-24 | Fever: temperature +1.4 °C, heart rate +14, worse sleep and mood |
| crew-02 | 35-41 | Stress week: low mood, short sleep, slower reactions, more lapses |
| crew-03 | 48-50 | Mild fatigue: small changes, none extreme on its own |
| crew-04 | none | Control |

In [ ]:
DAYS = 60
FIRST_MISSION_DAY = 83
START_DATE = date(2026, 9, 1)

CREW = [
    {"id": "crew-01", "name": "Maya Okafor", "role": "Flight engineer"},
    {"id": "crew-02", "name": "Tomas Reyes", "role": "Commander"},
    {"id": "crew-03", "name": "Lena Kowalski", "role": "Crew medical officer"},
    {"id": "crew-04", "name": "Kwame Asante", "role": "Mission specialist"},
]

# Day-to-day noise (standard deviation) for each indicator.
NOISE = {"systolic": 4.0, "diastolic": 3.0, "heartRate": 3.0, "glucose": 5.0,
         "temperature": 0.15, "sleepHours": 0.4, "mood": 0.35, "reactionMs": 12.0}

EVENTS = [
    {"crewId": "crew-01", "startDay": 22, "endDay": 24, "type": "fever",
     "changes": {"temperature": 1.4, "heartRate": 14, "sleepHours": -1.0, "mood": -1.0}},
    {"crewId": "crew-02", "startDay": 35, "endDay": 41, "type": "stress_week",
     "changes": {"mood": -1.5, "sleepHours": -1.3, "reactionMs": 45, "lapses": 4, "heartRate": 6, "systolic": 6}},
    {"crewId": "crew-03", "startDay": 48, "endDay": 50, "type": "mild_fatigue",
     "changes": {"sleepHours": -0.7, "reactionMs": 20, "mood": -0.7, "lapses": 1}},
]


def make_profile(rng):
    return {"systolic": rng.uniform(110, 122), "diastolic": rng.uniform(70, 79),
            "heartRate": rng.uniform(58, 70), "glucose": rng.uniform(84, 94),
            "temperature": rng.uniform(36.6, 36.9), "sleepHours": rng.uniform(6.8, 7.3),
            "mood": rng.uniform(3.8, 4.3), "reactionMs": rng.uniform(250, 290),
            "lapses": rng.uniform(0.5, 1.5)}


def generate(seed=2026):
    rng = np.random.default_rng(seed)
    crew_out = []
    for member in CREW:
        profile = make_profile(rng)
        member_events = [e for e in EVENTS if e["crewId"] == member["id"]]
        days = []
        for i in range(DAYS):
            d = i + 1
            v = {k: profile[k] + rng.normal(0, s) for k, s in NOISE.items()}
            v["sleepHours"] += -0.6 * i / (DAYS - 1)  # slow sleep drift
            lam = profile["lapses"]
            for e in member_events:
                if e["startDay"] <= d <= e["endDay"]:
                    for key, delta in e["changes"].items():
                        if key == "lapses":
                            lam += delta
                        else:
                            v[key] += delta
            days.append({
                "day": d, "missionDay": FIRST_MISSION_DAY + i,
                "date": (START_DATE + timedelta(days=i)).isoformat(),
                "systolic": round(v["systolic"]), "diastolic": round(v["diastolic"]),
                "heartRate": round(v["heartRate"]), "glucose": round(v["glucose"]),
                "temperature": round(v["temperature"], 1),
                "sleepHours": round(float(np.clip(v["sleepHours"], 2, 10)), 1),
                "mood": int(np.clip(round(v["mood"]), 1, 5)),
                "reactionMs": round(v["reactionMs"]), "lapses": int(rng.poisson(lam)),
            })
        crew_out.append({**member, "days": days})
    return crew_out


crew = generate(2026)
df = pd.DataFrame([{"crewId": c["id"], **d} for c in crew for d in c["days"]])
display(df.head(8))

In [ ]:
# Look at two crew members: Maya's fever and Tomas's stress week stand out clearly.
fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True)
plots = [("crew-01", "temperature", "Maya: temperature (°C)"), ("crew-01", "sleepHours", "Maya: sleep (h)"),
         ("crew-02", "reactionMs", "Tomas: reaction time (ms)"), ("crew-02", "mood", "Tomas: mood (1-5)")]
for ax, (cid, col, title) in zip(axes.flat, plots):
    g = df[df["crewId"] == cid]
    ax.plot(g["day"], g[col], marker=".", lw=1)
    for e in EVENTS:
        if e["crewId"] == cid:
            ax.axvspan(e["startDay"] - 0.5, e["endDay"] + 0.5, color="orange", alpha=0.25, label=e["type"])
    ax.set_title(title)
    ax.legend(loc="best", fontsize=8)
for ax in axes[1]:
    ax.set_xlabel("Day of data")
plt.tight_layout()
plt.show()

## 2. The baseline engine

This is a Python copy of `next-app/src/lib/baseline.js`, with the same rules and thresholds. Every indicator gets three checks each day:

1. **General range:** outside adult limits → watch or alert.
2. **Personal baseline:** z-score against the previous 14 days (needs at least 7). Watch at 2 SD, alert at 3.5 SD, and only in the direction that matters (short sleep is bad, long sleep isn't).
3. **Slow drift:** a moving average (EWMA) compared with the first two weeks. Watch if it stays 1 SD off for 5 days.

Alert-level readings are kept out of the baseline, so a fever doesn't become "normal". A day is raised with the astronaut if any indicator is at alert, two or more are at watch, or something is drifting.

The general ranges are approximate adult guidance and need cited sources before they're shown to users.

In [ ]:
INDICATORS = {
    "systolic":    dict(label="Systolic blood pressure", unit="mmHg", direction="high", minSd=3,
                        range=dict(watchHigh=130, alertHigh=140, watchLow=90, alertLow=80)),
    "diastolic":   dict(label="Diastolic blood pressure", unit="mmHg", direction="high", minSd=2.5,
                        range=dict(watchHigh=85, alertHigh=90, watchLow=60, alertLow=50)),
    "heartRate":   dict(label="Resting heart rate", unit="bpm", direction="both", minSd=2.5,
                        range=dict(watchHigh=100, alertHigh=120, watchLow=50, alertLow=40)),
    "glucose":     dict(label="Blood sugar (fasting)", unit="mg/dL", direction="both", minSd=4,
                        range=dict(watchHigh=100, alertHigh=126, watchLow=70, alertLow=54)),
    "temperature": dict(label="Body temperature", unit="°C", direction="high", minSd=0.15,
                        range=dict(watchHigh=37.5, alertHigh=38.0, watchLow=36.0, alertLow=35.5)),
    "sleepHours":  dict(label="Sleep", unit="h", direction="low", minSd=0.35, range=dict(watchLow=6.0, alertLow=4.5)),
    "mood":        dict(label="Mood", unit="/5", direction="low", minSd=0.5, range=dict(watchLow=2, alertLow=1)),
    "reactionMs":  dict(label="Reaction time", unit="ms", direction="high", minSd=10, range={}),
    "lapses":      dict(label="Reaction lapses", unit="per test", direction="high", minSd=1.5, range={}),
}
FEATURES = list(INDICATORS)
SETTINGS = dict(windowDays=14, minHistory=7, watchZ=2, alertZ=3.5, ewmaAlpha=0.3,
                driftZ=1, driftDays=5, minWatchCount=2)
RANK = {"normal": 0, "watch": 1, "alert": 2}


def worse(a, b):
    return a if RANK[a] >= RANK[b] else b


def mean_sd(values, min_sd):
    n = len(values)
    mean = sum(values) / n
    var = sum((v - mean) ** 2 for v in values) / max(n - 1, 1)
    return mean, max(var ** 0.5, min_sd)


def fmt(key, v):
    digits = 1 if key in ("temperature", "sleepHours") else 0
    unit = INDICATORS[key]["unit"]
    return f"{v:.{digits}f}{unit}" if unit.startswith("/") else f"{v:.{digits}f} {unit}"


def range_check(key, value):
    r, label = INDICATORS[key]["range"], INDICATORS[key]["label"]
    if "alertHigh" in r and value >= r["alertHigh"]:
        return "alert", f"{label} {fmt(key, value)} is at or above {fmt(key, r['alertHigh'])}"
    if "alertLow" in r and value <= r["alertLow"]:
        return "alert", f"{label} {fmt(key, value)} is at or below {fmt(key, r['alertLow'])}"
    if "watchHigh" in r and value >= r["watchHigh"]:
        return "watch", f"{label} {fmt(key, value)} is at or above {fmt(key, r['watchHigh'])}"
    if "watchLow" in r and value < r["watchLow"]:
        return "watch", f"{label} {fmt(key, value)} is below {fmt(key, r['watchLow'])}"
    return "normal", None


def bad_side(direction, z):
    return z if direction == "high" else -z if direction == "low" else abs(z)


def baseline_check(key, value, history):
    cfg = INDICATORS[key]
    if len(history) < SETTINGS["minHistory"]:
        return "normal", None, None
    mean, sd = mean_sd(history, cfg["minSd"])
    z = (value - mean) / sd
    bad = bad_side(cfg["direction"], z)
    status = "alert" if bad >= SETTINGS["alertZ"] else "watch" if bad >= SETTINGS["watchZ"] else "normal"
    if status == "normal":
        return status, z, None
    side = "above" if z > 0 else "below"
    return status, z, (f"{cfg['label']} {fmt(key, value)} is {abs(z):.1f} SD {side} your "
                       f"{len(history)}-day average of {fmt(key, mean)}")


def evaluate_series(days):
    clean = {k: [] for k in FEATURES}
    reference, ewma, drift_run = {}, {}, {k: 0 for k in FEATURES}
    results = []
    for i, reading in enumerate(days):
        indicators, overall = {}, "normal"
        for key in FEATURES:
            value, cfg = reading[key], INDICATORS[key]
            history = clean[key][-SETTINGS["windowDays"]:]
            r_status, r_reason = range_check(key, value)
            b_status, z, b_reason = baseline_check(key, value, history)
            d_status, d_reason = "normal", None
            if i == SETTINGS["windowDays"] and key not in reference:
                reference[key] = mean_sd(clean[key], cfg["minSd"])
            outlier = r_status == "alert" or b_status == "alert"
            if key not in ewma:
                ewma[key] = value
            elif not outlier:
                ewma[key] = SETTINGS["ewmaAlpha"] * value + (1 - SETTINGS["ewmaAlpha"]) * ewma[key]
            if key in reference:
                ref_mean, ref_sd = reference[key]
                dz = (ewma[key] - ref_mean) / ref_sd
                drift_run[key] = drift_run[key] + 1 if bad_side(cfg["direction"], dz) >= SETTINGS["driftZ"] else 0
                if drift_run[key] >= SETTINGS["driftDays"]:
                    d_status = "watch"
                    d_reason = (f"{cfg['label']} has drifted {'up' if dz > 0 else 'down'} from your "
                                f"mission-start average of {fmt(key, ref_mean)} for {drift_run[key]} days")
            status = worse(worse(r_status, b_status), d_status)
            indicators[key] = dict(value=value, status=status, range=r_status, baseline=b_status, drift=d_status,
                                   reasons=[x for x in (r_reason, b_reason, d_reason) if x])
            overall = worse(overall, status)
            if not outlier:
                clean[key].append(value)
        vals = indicators.values()
        attention = (any(x["status"] == "alert" for x in vals)
                     or sum(x["status"] == "watch" for x in vals) >= SETTINGS["minWatchCount"]
                     or any(x["drift"] != "normal" for x in vals))
        results.append(dict(day=reading["day"], overall=overall, attention=attention, indicators=indicators))
    return results


engine = {c["id"]: evaluate_series(c["days"]) for c in crew}

# What the astronaut would read on two event days:
for cid, day in [("crew-01", 22), ("crew-02", 36)]:
    r = engine[cid][day - 1]
    print(f"{cid}, day {day}: {r['overall'].upper()}")
    for k, v in r["indicators"].items():
        if v["reasons"]:
            print(f"   [{v['status']}] " + "; ".join(v["reasons"]))

## 3. Isolation Forest

Isolation Forest looks at all nine readings for a day together, and scores how unusual that combination is. That lets it catch days where nothing is extreme on its own.

- Each reading is first rescaled against **that person's own first 14 days**, because people have different normals.
- The model is trained only on those first 14 days, 56 rows across the crew, so days 15-60 are new to it.
- Each flag is explained by the two readings furthest from the person's normal.

In [ ]:
MIN_SD = {k: v["minSd"] for k, v in INDICATORS.items()}
MIN_SD["lapses"] = 1  # the model uses a slightly tighter floor than the app, as in the practice scripts
TRAIN_DAYS, FIRST_SCORED_DAY, RECOVERY_DAYS = 14, 15, 3


def personal_scaler(df):
    """Each person's mean and spread over their first 14 days. Save this with the model."""
    ref = df[df["day"] <= TRAIN_DAYS]
    return {cid: {f: (g[f].mean(), max(g[f].std(ddof=1), MIN_SD[f])) for f in FEATURES}
            for cid, g in ref.groupby("crewId")}


def standardise(df, scaler):
    z = df.copy()
    for f in FEATURES:
        z[f] = [(v - scaler[c][f][0]) / scaler[c][f][1] for c, v in zip(df["crewId"], df[f])]
    return z


def train_isolation_forest(df, contamination=0.03):
    scaler = personal_scaler(df)
    z = standardise(df, scaler)
    model = IsolationForest(n_estimators=300, contamination=contamination, random_state=0)
    model.fit(z[z["day"] <= TRAIN_DAYS][FEATURES])
    z["ifFlag"] = model.predict(z[FEATURES]) == -1
    z["ifScore"] = model.decision_function(z[FEATURES])
    z["why"] = z[FEATURES].abs().apply(
        lambda r: ", ".join(f"{k} {v:.1f} SD" for k, v in r.sort_values(ascending=False).head(2).items()), axis=1)
    return model, scaler, z


model, scaler, scored = train_isolation_forest(df)
display(scored[scored["ifFlag"] & (scored["day"] >= FIRST_SCORED_DAY)][["crewId", "day", "ifScore", "why"]])

## 4. Compare the three methods

The same rules apply to every method, and they were decided before looking at the results:

- **Scored days:** only days 15-60 count, since the model learns from days 1-14.
- **Caught:** an event is caught if any day inside it is flagged.
- **False alarm:** a flagged day outside every event and outside the 3 recovery days after one.
- **Range rules:** use the same day rule as the engine (any alert, or 2+ watch), applied to the range checks only.

In [ ]:
def method_flags(crew, contamination=0.03):
    df = pd.DataFrame([{"crewId": c["id"], **d} for c in crew for d in c["days"]])
    rows = []
    for c in crew:
        for r in evaluate_series(c["days"]):
            ind = r["indicators"].values()
            sleep = r["indicators"]["sleepHours"]
            others = [v for k, v in r["indicators"].items() if k != "sleepHours"] + (
                [sleep] if sleep["range"] != "normal" or sleep["baseline"] != "normal" else [])
            drift_only = r["attention"] and sleep["drift"] != "normal" and not (
                any(v["status"] == "alert" for v in others) or sum(v["status"] == "watch" for v in others) >= 2)
            rows.append(dict(crewId=c["id"], day=r["day"],
                             rangeFlag=any(x["range"] == "alert" for x in ind) or sum(x["range"] == "watch" for x in ind) >= 2,
                             engineFlag=r["attention"], sleepDriftOnly=drift_only))
    flags = pd.DataFrame(rows)
    _, _, z = train_isolation_forest(df, contamination)
    return flags.merge(z[["crewId", "day", "ifFlag"]], on=["crewId", "day"])


def score(flags, col):
    s = flags[flags["day"] >= FIRST_SCORED_DAY]
    caught = {}
    for e in EVENTS:
        win = s[(s["crewId"] == e["crewId"]) & s["day"].between(e["startDay"], e["endDay"])]
        caught[e["type"]] = (int(win[col].sum()), len(win))
    protected = pd.Series(False, index=s.index)
    for e in EVENTS:
        protected |= (s["crewId"] == e["crewId"]) & s["day"].between(e["startDay"], e["endDay"] + RECOVERY_DAYS)
    fa = s[col] & ~protected
    drift_fa = int((fa & s["sleepDriftOnly"]).sum()) if col == "engineFlag" else 0
    return caught, int(fa.sum()), drift_fa, int((~protected).sum())


METHODS = {"(a) Earth range rules": "rangeFlag", "(b) Baseline engine": "engineFlag", "(c) Isolation Forest": "ifFlag"}

flags = method_flags(crew)
table = []
for name, col in METHODS.items():
    caught, fa, drift_fa, eligible = score(flags, col)
    row = {"Method": name}
    row.update({ev: f"{'yes' if hit else 'no'} ({hit}/{n} days)" for ev, (hit, n) in caught.items()})
    row["False alarm days"] = f"{fa} of {eligible} ({fa / eligible:.0%})"
    row["of which sleep drift"] = drift_fa
    table.append(row)
print("One dataset (seed 2026):")
display(pd.DataFrame(table))

### One dataset can flatter a method, so test on 20 fresh ones

The settings were tuned on seed 2026, so that result is too optimistic. On it, Isolation Forest looks nearly perfect. This cell generates 20 new datasets the settings never saw and averages the results. These are the numbers to quote.

In [ ]:
rows = []
for seed in range(3001, 3021):
    f = method_flags(generate(seed))
    for name, col in METHODS.items():
        caught, fa, drift_fa, eligible = score(f, col)
        rows.append({"Method": name, **{ev: hit > 0 for ev, (hit, n) in caught.items()},
                     "fa": fa / eligible, "fa_excl_drift": (fa - drift_fa) / eligible})
res = pd.DataFrame(rows).groupby("Method").mean()
summary = pd.DataFrame({
    "Fever caught": res["fever"].map("{:.0%}".format),
    "Stress week caught": res["stress_week"].map("{:.0%}".format),
    "Mild fatigue caught": res["mild_fatigue"].map("{:.0%}".format),
    "False alarm days": res["fa"].map("{:.0%}".format),
    "False alarms, not counting sleep drift": res["fa_excl_drift"].map("{:.0%}".format),
})
print("Averages over 20 fresh datasets (seeds 3001-3020):")
display(summary)

**What this shows:**

- **Earth ranges miss most of what matters.** The readings stay inside normal adult limits, so the range rules catch only about 60% of stress weeks and 5% of mild fatigue.
- **The baseline engine and Isolation Forest perform about the same** on single-day false alarms (7% and 9%).
- **The drift check is the noisy part**, and the first thing to tune.

## 5. Export for the app

- **`crew.json`** goes to `next-app/src/data/` and feeds the dashboard.
- **`model.joblib`** holds the trained model plus each person's scaler, only needed if we add a live scoring API later. The scaler must travel with the model: live scoring has to rescale readings exactly as training did.

In [ ]:
import joblib

with open("crew.json", "w") as fh:
    json.dump({"note": "Synthetic data for a fictional crew. Not real astronaut records.",
               "seed": 2026, "crew": crew}, fh, indent=1)
joblib.dump({"model": model, "scaler": scaler, "features": FEATURES}, "model.joblib")
print("Saved crew.json and model.joblib")

try:  # Colab only: download both files to your computer
    from google.colab import files
    files.download("crew.json")
    files.download("model.joblib")
except ImportError:
    pass

## Limitations

- **Synthetic data, not real astronauts.** It's built from published averages, and the noise model is ours.
- **The planted events are our guesses** at what illness and stress look like in these readings.
- **The general ranges are approximate** adult guidance and need cited sources.
- **This is decision support, not a medical device.**

**AI use:** this notebook was drafted with Claude (Anthropic) and reviewed and run by the team.